### Generate SMILES with LLAMA 3.2-3B

In [ ]:
!pip install -U transformers

In [ ]:
from huggingface_hub import login, whoami

login(new_session=False)
#print(whoami())

### Tokenizer and model setup

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_name = "meta-llama/Llama-3.2-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)


tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/1.46G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

In [ ]:
def generate_llama_smiles(prompt, n=30, max_new_tokens=800):
    messages = [
        {"role": "system", "content": "You are a chemical SMILES generator."},
        {"role": "user", "content": prompt},
    ]

    # Let the tokenizer directly return a dictionary: input_ids + attention_mask
    encoded = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_tensors="pt",
        return_dict=True,
    )

    # Move all tensors to model.device
    encoded = {k: v.to(model.device) for k, v in encoded.items()}

    # pad_token may be None; fall back to eos_token if needed
    pad_id = tokenizer.pad_token_id or tokenizer.eos_token_id

    outputs = model.generate(
        **encoded,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=1.0,
        top_p=0.95,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=pad_id,
    )

    # Keep only the newly generated part (remove the prompt)
    generated = outputs[0][encoded["input_ids"].shape[-1]:]
    text = tokenizer.decode(generated, skip_special_tokens=True)

    # ====== Post-processing: split by lines & clean ======
    lines = [x.strip() for x in text.split("\n") if x.strip()]
    cleaned = []

    for line in lines:
        # Remove leading numbering, e.g. "1. CCO"
        if line[0].isdigit() and "." in line[:3]:
            line = line.split(".", 1)[1].strip()
        cleaned.append(line)

    return cleaned[:n]


In [ ]:
# Prepare a few-shot high-conductivity prompt
# Generate high-conductivity SMILES
high_prompt_gpt4o = """
You are a polymer electrolyte design assistant.
Generate 100 unique polymer-like molecules in SMILES format with HIGH ionic conductivity.

Requirements:
- Each molecule must contain exactly one [Cu] and one [Au].
- Include polar groups (O, N, CO).
- Must be valid SMILES, one per line.
- Must not repeat the examples.

Examples:
NC(=O)CSCC(CO[Cu])OC(=O)[Au]
CCN(CCO[Cu])CCC(CO)OC(=O)[Au]
OCNCCC(COC[Cu])COC(=O)[Au]

Now generate 100 new high-conductivity polymer molecules.
Please generate a list of 100 SMILES.
Output only one SMILES per line.
"""


In [ ]:
# Generate low-conductivity SMILES
low_prompt_gpt4o = """
You are a SMILES generator for polymer electrolytes.

Task:
Generate 100 VALID SMILES strings that have LOW ionic conductivity.

Hard constraints (must ALL be satisfied):
1. Each SMILES must contain EXACTLY one [Cu] and one [Au].
2. The backbone must be HYDROPHOBIC:
   - Use mainly C and H atoms.
   - MAY include at most ONE oxygen (O) OR ONE nitrogen (N), but not both.
3. MUST NOT contain more than one polar group (=O or -O- or -N-).
4. No whitespace, no numbering, no comments.
5. One SMILES per line.
6. Return ONLY SMILES lines.

Soft constraints (increase likelihood of low conductivity):
- Prefer long aliphatic carbon chains.
- Keep heteroatoms near the chain ends.

Examples (valid patterns):
CCCCCCC(C)CCO[Cu]CCC[Au]
CCC(C)CCCCCCC(=O)C[Cu]C[Au]
CCCCCCCCCN[Cu]CCCC(=O)[Au]

Now generate 100 new low-conductivity polymer molecules.
Please generate a list of 100 SMILES.
Output only one SMILES per line.
"""

### Generation and post processing

In [ ]:
import re

def clean_gpt4o_output(raw_list):
    cleaned = []

    for line in raw_list:
        line = line.strip()
        if not line:
            continue

        # 1) Remove leading numbering, e.g. "10. ", "3. "
        #    Pattern: optional leading spaces + digits + dot + optional spaces
        line = re.sub(r'^\s*\d+\.\s*', '', line)

        # 2) Must contain both Cu and Au (this constraint is kept)
        if "[Cu]" not in line or "[Au]" not in line:
            continue

        # 3) Filter out lines that are clearly not SMILES (contain long English words)
        #    Since the prompt already enforces “SMILES only”,
        #    this case should be rare
        if re.search(r"[A-Za-z]{3,}", line) and not re.search(r"[A-Z][a-z]?", line):
            continue

        # 4) Disallow spaces, commas, semicolons, etc.
        if re.search(r"[ ,:;]", line):
            continue

        # 5) Basic character check: only allow atoms, digits, parentheses, and bond symbols
        if not re.match(r"^[A-Za-z0-9\[\]\(\)=#]+$", line):
            continue

        cleaned.append(line)

    return cleaned

In [ ]:
# llama_high_results = generate_llama_smiles(high_prompt_gpt4o, n=100)
# llama_high_results[:10]


["Here's the list of 100 unique polymer-like molecules in SMILES format with high ionic conductivity:",
 'C(COO)(CO[Cu])CCC(CO)OC(=O)[Au]',
 'CCNC(CCO[Cu])OCC(CO)OCC(=O)[Au]',
 'CCCN(CCO[Cu])CCO(CO)OC(=O)[Au]',
 'OCNCC(C(CO)OC[Cu])COCC(=O)[Au]',
 'CC(CO)OCC(COO)(CO[Cu])CCC(=O)[Au]',
 'COCC(COO)[Au]NC=CSCC(CO[Cu])COCC(=O)',
 'CCC(CO)OC(=O)[Au]CCN(CCO[Cu])OCC(=O)',
 'CCCC(CO)CC(COO)[Au]NC=CSCC(CO[Cu])CO',
 'CC(CO)OCC(COO)(CO[Cu])CCC(CO)OC(=O)[Au]']

In [ ]:
target = 100          # Desired final number of unique SMILES
max_rounds = 20       # Maximum number of attempts to avoid an infinite loop

all_high = []

for i in range(max_rounds):
    if len(dict.fromkeys(all_high)) >= target:
        break

    batch = generate_llama_smiles(high_prompt_gpt4o, n=100, max_new_tokens=800)
    cleaned = clean_gpt4o_output(batch)
    all_high.extend(cleaned)

    print(f"Round {i+1}: cleaned = {len(cleaned)}, "
          f"unique so far = {len(dict.fromkeys(all_high))}")

# Final deduplication
llama_high_results = list(dict.fromkeys(all_high))
print("Final unique HIGH-conductivity SMILES:", len(llama_high_results))

Round 1: cleaned = 22, unique so far = 18
Round 2: cleaned = 27, unique so far = 44
Round 3: cleaned = 39, unique so far = 68
Round 4: cleaned = 15, unique so far = 81
Round 5: cleaned = 22, unique so far = 103
Final unique HIGH-conductivity SMILES: 103


In [ ]:
# llama_low_results = generate_llama_smiles(low_prompt_gpt4o, n=100)
# llama_low_results[:10]

['Here is a list of 100 valid SMILES strings that meet the hard and soft constraints, representing low ionic conductivity polymer electrolytes:',
 'CCCC[Cu]CCC[O]CCC[Au]',
 'CCCC[Cu]CCC[C]CCO[Au]',
 'C(C)C[Cu]CCC[C]CCC[Au]',
 'CCCC[Cu]CCC[C][O]CC[Au]',
 'C[C]C[Cu]CCCCCC[Au]',
 'CCCC[Cu]CCC[N]CCC[Au]',
 'CCCC[Cu]CCC[C]CCC[C]CC[Au]',
 'CCCC[Cu]CCC[C]C(=O)[Au]',
 'C[C]C[Cu]CCC[C]CCC[C]CC[Au]']

In [ ]:
target = 100
max_rounds = 20

all_low = []

for i in range(max_rounds):
    if len(dict.fromkeys(all_low)) >= target:
        break

    batch = generate_llama_smiles(low_prompt_gpt4o, n=100, max_new_tokens=800)
    cleaned = clean_gpt4o_output(batch)
    all_low.extend(cleaned)

    print(f"Round {i+1}: cleaned = {len(cleaned)}, "
          f"unique so far = {len(dict.fromkeys(all_low))}")

llama_low_results = list(dict.fromkeys(all_low))
print("Final unique LOW-conductivity SMILES:", len(llama_low_results))


Round 1: cleaned = 11, unique so far = 11
Round 2: cleaned = 13, unique so far = 20
Round 3: cleaned = 6, unique so far = 24
Round 4: cleaned = 10, unique so far = 32
Round 5: cleaned = 19, unique so far = 44
Round 6: cleaned = 0, unique so far = 44
Round 7: cleaned = 31, unique so far = 72
Round 8: cleaned = 0, unique so far = 72
Round 9: cleaned = 28, unique so far = 100
Final unique LOW-conductivity SMILES: 100


In [ ]:
import json

with open("../../data/generated/llama/llama_high_clean.json", "w") as f:
    json.dump(llama_high_results, f)

with open("../../data/generated/llama/llama_low_clean.json", "w") as f:
    json.dump(llama_low_results, f)
